In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    return phonetic

In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16682/18310 [04:22<00:16, 99.20it/s] c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:41<00:00, 64.97it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [6]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [9]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,None,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,None,ERROR: argument of type 'NoneType' is not iter...
18279,ไอศวรรย์,ʔaj˧.sa˨˩.wan˩˩˦,1,ʔaj˧.saʔ˨˩.wan˨˥,None,ERROR: Could not align 'ไอศวรรย์' with 'ʔaj˧.s...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [10]:
errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
6657,บาท,baːt̚˨˩.,3,baːt˨˩.,None,ERROR: argument of type 'NoneType' is not iter...
14251,เกาะอีสเตอร์,kɔʔ˨˩.ʔiːs˦˥.tɤː˥˩,1,kɔʔ˨˩.ʔiːs˦˥.tɤː˦˩,None,ERROR: argument of type 'NoneType' is not iter...
8759,ม,mɔː˧,2,mɔː˧,None,ERROR: Could not align 'ม' with 'mɔː˧'
15183,เบลารุส,beː˧.laː˧.rus˦˥,2,beː˧.laː˧.rus˦˥,None,ERROR: argument of type 'NoneType' is not iter...
5429,ทาส,tʰaːt̚˥˩.sa˨˩.,2,tʰaːt˦˩.saʔ˨˩.,None,ERROR: argument of type 'NoneType' is not iter...
17717,โอลีฟ,ʔoː˧.liːf˦˥,1,ʔoː˧.liːf˦˥,None,ERROR: argument of type 'NoneType' is not iter...
16799,แฟรงก์เฟิร์ต,frɛŋ˧.fɤt̚˨˩,2,frɛŋ˧.fɤt˨˩,None,ERROR: Could not align 'แฟรงก์เฟิร์ต' with 'fr...
6762,บุญ,bun˧.ja˦˥.,3,bun˧.jaʔ˦˥.,None,ERROR: argument of type 'NoneType' is not iter...
5850,นอร์ทดาโคตา,nɔt̚˦˥.daː˧.kʰoː˧.taː˥˩,1,nɔt˦˥.daː˧.kʰoː˧.taː˦˩,None,ERROR: Could not align 'นอร์ทดาโคตา' with 'nɔt...
17058,แองกวิลลา,ʔɛŋ˧.kwil˧.laː˥˩,1,ʔɛŋ˧.kwil˧.laː˦˩,None,ERROR: argument of type 'NoneType' is not iter...
